# Handwritten Digits

Can a computer read your handwriting? In this notebook you draw a digit from 0 to 9 with your mouse or finger, and a small neural network tries to work out which digit it is. It guesses again with every stroke you add, so you can watch it change its mind as you draw.

The network was trained on **MNIST**, a famous collection of 70,000 handwritten digits. For decades, MNIST has been the first dataset people use to learn how machines recognize images.

## Loading the model

The trained network is attached to this notebook as a file called `mnist.onnx` (click the paperclip icon in the top right to see it). ONNX stands for Open Neural Network eXchange, a popular format for sharing trained models. The file is tiny, about 26 KB, which is smaller than most photos.

We load it with the `dnn` (deep neural network) module in OpenCV.

In [ ]:
import cv2

net = cv2.dnn.readNetFromONNX('/attachments/mnist.onnx')
print("Neural network loaded")

## Getting your drawing ready for the model

The model can't look at your drawing directly. Every image in MNIST is a tiny **28 x 28** grid of pixels, with a white digit on a black background, so that is the only kind of picture the model understands.

MNIST digits were also tidied up in the same way: each one was scaled to fit inside a 20 x 20 box, then placed so its "center of mass" sits in the middle of the grid. The `prepare` function below does the same to your drawing. That is why it doesn't matter whether you draw big or small, or in a corner of the pad.

Expand the cell if you want to see how it works!

In [ ]:
import numpy as np

def prepare(drawing):
  """Turn the drawing (a 2D array, 0 = empty, 255 = ink) into the 28x28
  image the model expects. Returns None if nothing has been drawn."""
  # 1. Find the part of the pad that has ink on it, and cut it out.
  rows, columns = np.nonzero(drawing)
  if len(rows) == 0:
    return None
  digit = drawing[rows.min():rows.max() + 1, columns.min():columns.max() + 1]

  # 2. Shrink it so the longer side is 20 pixels, like the MNIST digits.
  scale = 20 / max(digit.shape)
  width = max(1, round(digit.shape[1] * scale))
  height = max(1, round(digit.shape[0] * scale))
  digit = cv2.resize(digit, (width, height), interpolation=cv2.INTER_AREA)

  # 3. Place it on a 28x28 grid so its center of mass (the balance point
  #    of the ink) lands in the middle.
  moments = cv2.moments(digit)
  if moments["m00"] == 0:
    return None
  center_x = moments["m10"] / moments["m00"]
  center_y = moments["m01"] / moments["m00"]
  shift = np.float32([[1, 0, 13.5 - center_x], [0, 1, 13.5 - center_y]])
  return cv2.warpAffine(digit, shift, (28, 28))

## Asking the model

Now for the neural network itself. We hand it the 28 x 28 image, and it gives back ten scores, one for each digit from 0 to 9. The higher a score, the more the image looks like that digit to the model.

Raw scores are hard to read, so the last two lines turn them into **confidences** that add up to 100%. This step is called *softmax*.

In [ ]:
def predict(image):
  """Run the model on a 28x28 image. Returns ten confidences (0 to 1),
  one for each digit."""
  blob = cv2.dnn.blobFromImage(image, scalefactor=1 / 255.0)
  net.setInput(blob)
  scores = net.forward()[0]
  exponents = np.exp(scores - scores.max())
  return exponents / exponents.sum()

## Showing the results

The next cell contains the code that draws everything on the canvas: the drawing pad, the 28 x 28 image the model sees, its prediction, and a bar for each digit's confidence. It is ordinary canvas drawing code. Expand it if you are curious!

In [ ]:
PAD = (10, 10, 320, 320)          # where you draw: x, y, width, height
CLEAR_BUTTON = (10, 340, 320, 30)
INK = "#201E1D"
GREEN = "#56BC6C"
GREY = "#C9C5C1"

def show(canvas, image, confidences):
  """Redraw the pad's background and everything to the right of it."""
  ctx = canvas.get_context("2d")
  ctx.clear_rect(0, 0, 600, 380)
  ctx.text_align = "center"

  # The pad and the Clear button. Your strokes sit on a layer above this.
  ctx.fill_style = INK
  ctx.fill_rect(*PAD)
  ctx.fill_rect(*CLEAR_BUTTON)
  ctx.fill_style = "#ffffff"
  ctx.font = "bold 14px sans-serif"
  ctx.fill_text("Clear", 170, 360)
  if image is None:
    ctx.fill_style = "#8a8580"
    ctx.font = "18px sans-serif"
    ctx.fill_text("Draw a digit here", 170, 175)

  # What the model sees: the 28x28 image, each pixel drawn as a 5x5 block.
  ctx.fill_style = INK
  ctx.font = "12px sans-serif"
  ctx.text_align = "left"
  ctx.fill_text("What the model sees", 350, 22)
  ctx.fill_text("Prediction", 510, 22)
  ctx.fill_text("Confidence", 350, 196)
  canvas.draw_pixels(np.zeros((28, 28)) if image is None else image, 350, 30, 140, 140)

  # The prediction: the digit with the highest confidence.
  best = None if confidences is None else int(confidences.argmax())
  ctx.font = "bold 96px sans-serif"
  ctx.text_align = "center"
  ctx.fill_text("?" if best is None else str(best), 548, 135)

  # One bar per digit.
  ctx.font = "13px sans-serif"
  for digit in range(10):
    y = 204 + digit * 17
    confidence = 0 if confidences is None else float(confidences[digit])
    ctx.fill_style = INK
    ctx.text_align = "left"
    ctx.fill_text(str(digit), 350, y + 12)
    ctx.text_align = "right"
    ctx.fill_text(f"{confidence:.0%}", 400, y + 12)
    ctx.fill_style = GREEN if digit == best else GREY
    ctx.fill_rect(408, y + 2, 140 * confidence, 12)

def is_clear_button(x, y):
  left, top, width, height = CLEAR_BUTTON
  return left <= x <= left + width and top <= y <= top + height

## Draw a digit!

Time to put it all together. `canvas.on_draw` lets you draw on the pad, and calls our `update` function every time your drawing changes. Each update follows the same three steps: **prepare** the drawing, **predict** the digit, and **show** the results.

Run the cell, then draw a digit on the black pad. Click **Clear** to start again, and press the Stop button when you are done.

In [ ]:
from codetto import graphics2d

canvas = graphics2d.canvas(600, 380, scale=True)

def update():
  image = prepare(canvas.get_drawing())
  confidences = None if image is None else predict(image)
  show(canvas, image, confidences)

def handle_click(x, y):
  if is_clear_button(x, y):
    canvas.clear_drawing()
    update()

canvas.on_draw(update, line_width=22, color="#ffffff", area=PAD)
canvas.on_click(handle_click)

update()
canvas.run()

## Things to try

- Draw a **1**, then add a stroke across the top to turn it into a **7**. Watch the bars change as you draw.
- Draw a **4** with a closed top. Does the model think it is a **9**? Look at how similar they are in the 28 x 28 image.
- Draw a small digit in one corner of the pad. The model still sees it large and centered. Which function is responsible for that?
- Draw something that isn't a digit at all, like a letter or a smiley face. The model always picks a digit, because digits are all it knows. How confident is it?
- In the last code cell, change `line_width=22` to `line_width=6` and run it again. Thin strokes almost disappear when they are shrunk to 28 x 28, and the model gets worse.

# Check Your Understanding